## 04 - TME Deconvolution Plan (XDec + CIBERSORTx)

### Aim
* Plan a tumor microenvironment (TME) deconvolution pass on the QC'd TCGA-PAAD TPM matrix, combining XDec (reference-optional cancer cell state discovery) and CIBERSORTx (reference-based immune cell fraction estimation). **Planning only - no code is executed in this notebook.**

### Input
* `data/processed/TCGA-PAAD_TPM_raw.tsv` and `TCGA-PAAD_sample_metadata.tsv` (notebook 02).
* `data/qc/flagged_samples.tsv`, `data/qc/batch_effect_check.txt` (notebook 03) - the open sample-inclusion and batch-effect questions this deconvolution run is meant to help resolve.

### Output
* This planning document: tool introductions, an R-vs-Python recommendation, and a numbered implementation plan. No data files are produced here.

### Steps
* See "Planned pipeline" section below for the full numbered sequence.

### Result
* N/A - this is a planning document, nothing has been run yet.

### Conclusion
* Recommend doing both tools in **R**, for the reasons laid out below. The plan is ready to implement as a follow-up notebook (`05_xdec_deconvolution.ipynb` / `06_cibersortx_deconvolution.ipynb`) once the open decisions at the end of this notebook are resolved.


## Why two tools, not one

XDec and CIBERSORTx answer different questions and are complementary rather than redundant:

* **XDec** is *reference-optional* and data-driven. It doesn't require a pre-specified catalog of cell types - it discovers recurrent transcriptional **states** directly from the bulk tumor profiles themselves, including novel cancer-cell states that don't correspond to any single-cell reference. This is the right tool for the cancer-cell and stromal-state side of the question (e.g. Basal-like vs. Classical PDAC cancer-cell states, CAF vs. adipocyte stromal states) - exactly what the XDec-SM and XDec-CHI papers in this project's `papers/` folder were built to do.
* **CIBERSORTx** is *reference-based*. It requires a signature matrix of known, purified cell-type expression profiles (e.g. 22 immune cell types) and fits bulk mixtures as combinations of those fixed references. This is the right tool for well-characterized **immune cell** composition, where decades of sorted-reference data (the LM22 signature) give it strong validation and fine-grained immune subtyping (naive vs. memory T cells, M1 vs. M2 macrophages, etc.) that a reference-optional method like XDec isn't designed to resolve on its own.

Running both and cross-checking where they overlap (stromal/immune proportions) mirrors exactly what the XDec-SM paper itself did as a validation step (Fig 2/3: XDec-SM vs. CIBERSORTx, EPIC, xCell, Kassandra, InfiniumPurify, TIMER).

## XDec: package and algorithm

**What it is:** XDec-SM / XDec-CHI (Murillo et al. 2023, *PLOS Comp Bio*; Lewis/LaPlante et al. 2022, *iScience* - both in `papers/`) is an extension of **EDec** (the authors' earlier DNA-methylation deconvolution method), adapted for bulk RNA-seq.

**Algorithm (3 stages, as implemented):**
1. **Stage 0 (optional):** identify an informative gene set - genes differentially expressed between cell types/states in a reference (scRNA-seq pseudobulks, sorted cell lines, or a literature gene signature like PAM50). References are used *only* to pick which genes matter, never for quantitative values - this is what makes the method "reference-optional."
2. **Stage 1:** constrained matrix factorization via quadratic programming (`V ≈ W x H`) on gene expression values rescaled to 0-1 per gene (`y = 1 - e^{-ax}`, `a = 1/max{x}`, computed over the sample set being deconvolved). Produces per-sample cell-type/state **proportions** and the corresponding (rescaled) cell-type **expression profiles**, with the number of states (k) chosen by a stability procedure (3 random-subsample replicates across k=3..10/12, picking the k where results stay stable).
3. **Stage 2:** non-negative least squares (also via quadratic programming) using the Stage-1 proportions against the *untransformed* bulk profiles, to recover cell-type-specific expression profiles on the original (count/TPM) scale.

**XDec-CHI adds:** a sliding-window deconvolution across samples ranked by a gene of interest, repeating Stage 2 on overlapping windows of ≥40 samples to get cell-type-*specific* gene-gene correlations - this is how the PDAC paper found cancer-cell `MSLN` correlating with stromal `IFNGR1`, for example. This is the method for discovering intracellular/heterotypic interactions, not just getting a one-shot proportions table.

**Package:** R only. `https://github.com/BRL-BCM/XDec_SM` and `https://github.com/BRL-BCM/XDec_CHI` (the PDAC-specific sliding-window variant), both from the Milosavljevic lab (Baylor College of Medicine) - the same group behind `EDec`. No Python implementation exists.

## CIBERSORTx: package and algorithm

**What it is:** CIBERSORTx (Newman et al. 2019, *Nat Biotechnol*) is the successor to the original CIBERSORT, from the Alizadeh/Newman lab (Stanford).

**Algorithm:** support vector regression (nu-SVR) - bulk expression for each sample is modeled as a linear mixture of reference cell-type profiles, and nu-SVR fits the mixing proportions while being robust to noise and to the presence of cell types not in the reference (unlike ordinary least squares). Three modes relevant here:
* **Fractions mode:** per-sample relative (or, with normalization, absolute) proportions of each reference cell type.
* **High-Resolution (Hi-Res) mode:** cell-type-specific gene expression profiles per sample - the CIBERSORTx analogue of XDec's Stage 2 output.
* **Create Signature Matrix:** builds a custom reference signature matrix from scRNA-seq or sorted-bulk input, instead of using the default.

**Default reference:** **LM22**, a 22-immune-cell-type signature matrix built from sorted leukocyte expression profiles (not tumor-derived) - this is the standard reference most published CIBERSORTx TME studies use, including the XDec-SM comparison.

**Package / access model:** CIBERSORTx is **not** a CRAN or PyPI package - it's distributed as a **Docker container** (`cibersortx/fractions`, `cibersortx/hires`), run from the command line against a registration token obtained from `https://cibersortx.stanford.edu`. It is language-agnostic infrastructure: any language can prepare the input TSV and call `docker run`, and any language can read the output TSV back in. There's also a web interface for smaller/one-off jobs that avoids Docker entirely.

## R or Python?

**Recommendation: R, for both tools.**

* **XDec has no choice to make** - it's R-only. Since this project is explicitly built around reproducing/extending the XDec-SM and XDec-CHI methods, R is required regardless of preference.
* **CIBERSORTx is language-agnostic** (a Docker CLI tool) - R vs. Python only matters for the pre/post-processing around it, not the deconvolution itself. Doing that pre/post-processing in R keeps everything in one environment: the same TPM matrix, the same gene-symbol conventions, the same `pixi` R environment already set up for notebooks 00-03, with no format conversion or duplicate data-loading code in a second language.
* If CIBERSORTx's Docker/token registration is undesirable (external account, local Docker required), the **free R alternative is `immunedeconv`** (wraps xCell, EPIC, quanTIseq, MCPcounter, ESTIMATE, TIMER with one consistent interface, no registration needed) or **`IOBR`** (broader TME toolkit built specifically for TCGA-style bulk RNA-seq analysis, bundling the same methods plus signature scoring and visualization). Both are pure-R, install via `BiocManager`/`install_github`, and were already noted as optional packages in notebook `00_install_packages.ipynb`. CIBERSORTx itself is the one method from that comparison set that genuinely needs registration + Docker, since the authors restrict the trained SVR model behind a token system - there's no way around that regardless of language.

## Planned pipeline

1. **Resolve sample inclusion first.** The 8 PCA-outlier and 4 sex-mismatch samples from notebook 03 are still an open question. Plan: run deconvolution on the *full* 182-sample matrix first (don't pre-exclude), then use the resulting proportions to answer the question notebook 03 raised - do the 8 PCA outliers' proportions track `tumor_grade` (keep them) or `TSS` with no biological story (exclude / batch-correct)? This also resolves the step-8 batch-effect question from the same notebook. Decide sample exclusion *after* this check, not before.

2. **XDec Stage 0 - informative gene set.** Two options: (a) re-derive via EDec's one-vs-rest differential expression on a public PDAC scRNA-seq reference (the XDec-CHI paper used Peng et al. 2019, GSA: CRA001160), matching the paper's own pipeline; or (b) reuse the XDec-CHI paper's published 283-gene informative set (Table S3, in the paper's Supplemental Information - not currently downloaded locally, would need to be pulled from the journal's supplementary files). Option (b) is faster and directly comparable to the published PDAC results; option (a) is more self-contained but requires sourcing and processing the scRNA-seq reference first.

3. **XDec Stage 1.** Run the constrained matrix factorization on the full TPM matrix (all genes present after intersecting with the Stage-0 informative set), with the stability procedure (3 subsample replicates, k = 3..10) to pick the number of cell types/states.

4. **XDec Stage 2.** Recover untransformed, per-cell-type expression profiles via NNLS using the Stage-1 proportions.

5. **(Optional) XDec-CHI sliding-window analysis.** Only needed if the goal extends beyond proportions to specific gene-gene interaction discovery (the PDAC paper's headline result, e.g. cancer-cell `MSLN` vs. stromal `IFNGR1`) - requires ≥40-sample windows, comfortably met by this 182-sample cohort.

6. **CIBERSORTx Fractions mode.** Run the same TPM matrix through CIBERSORTx (Docker, LM22 default signature) to get independent immune-cell proportions - register for a token, confirm Docker is available locally, run `cibersortx/fractions` against the TPM matrix.

7. **Cross-validate.** Compare XDec's summed stromal/immune proportions against CIBERSORTx's, the same comparison the XDec-SM paper itself performed against CIBERSORTx/EPIC/xCell/Kassandra (their Fig 2/3) - a sanity check that both tools broadly agree on the non-cancer compartment, even though XDec's cancer-cell-state resolution is the point CIBERSORTx can't match.

8. **Resolve the open QC questions.** Use step 1's output to settle the PCA-outlier and batch-effect questions left open in notebook 03.

9. **Save outputs.** Proportions and cell-type-specific profiles from both tools, written to a new `data/deconvolution/` directory (not yet created) alongside a short README documenting which informative gene set and which CIBERSORTx reference/mode were used, for reproducibility.

## Open decisions before implementing

* **Gene set for XDec Stage 0:** re-derive from scRNA-seq (matches the paper's method, more work) vs. reuse the published 283-gene PDAC list (faster, needs the supplementary table pulled from the journal).
* **CIBERSORTx access:** willing to register for a token and run Docker locally, or prefer to skip CIBERSORTx and use the registration-free R alternatives (`immunedeconv` / `IOBR`) for the reference-based comparison instead?
* **Custom vs. default CIBERSORTx signature:** stick with LM22 (blood-derived, well-validated, used in the XDec-SM paper's own comparison) or build a PDAC-specific signature matrix from the same scRNA-seq reference being considered for XDec Stage 0 (more tumor-relevant, but another processing step and another thing to validate)?
* **Sample set:** confirmed above to decide *after* step 1's post-hoc check, not before - flagged here as a reminder not to skip that check under time pressure.